# 02 - 数据加载与预处理

## 学习目标
- 掌握 sklearn 内置数据集的加载方式
- 理解数据预处理的必要性
- 学会使用 StandardScaler、LabelEncoder 等工具
- 掌握 train_test_split 的正确用法

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

## 1. sklearn 内置数据集

In [2]:
# 分类数据集
iris = datasets.load_iris()
print(f'Iris 数据集: {iris.data.shape}, 类别: {iris.target_names}')

breast_cancer = datasets.load_breast_cancer()
print(f'Breast Cancer: {breast_cancer.data.shape}, 类别: {breast_cancer.target_names}')

digits = datasets.load_digits()
print(f'Digits: {digits.data.shape}, 类别数: {len(digits.target_names)}')

# 回归数据集
california = datasets.fetch_california_housing()
print(f'California Housing: {california.data.shape}')

Iris 数据集: (150, 4), 类别: ['setosa' 'versicolor' 'virginica']
Breast Cancer: (569, 30), 类别: ['malignant' 'benign']
Digits: (1797, 64), 类别数: 10
California Housing: (20640, 8)


在机器学习和 Python（NumPy）中，`(150, 4)` 表示特征矩阵 `iris.data` 的**形状（Shape）**，格式为 **`(样本数, 特征数)`**（即 **150 行、4 列** 的二维数组）：

---

### 1. 各个数字的具体含义

* **`150`（行数 / 样本量）：**
  * 数据集中一共包含 **150 个样本**（即采集了 150 朵鸢尾花的数据）。
  * 对应 3 种鸢尾花类别（`setosa`、`versicolor`、`virginica`），每个类别各有 50 朵。

* **`4`（列数 / 特征数）：**
  * 针对每朵鸢尾花，测量记录了 **4 个数值特征（属性）**。
  * 你可以通过 `iris.feature_names` 查看具体的特征名称：
    1. `sepal length (cm)`：花萼长度
    2. `sepal width (cm)`：花萼宽度
    3. `petal length (cm)`：花瓣长度
    4. `petal width (cm)`：花瓣宽度

---

### 2. 对应的标签（Target）

与特征矩阵 `iris.data` 对应的标签是 `iris.target`：
* `iris.target.shape` 是 **`(150,)`**，表示一维数组，里面包含 150 个整数（0、1、2），分别对应上面 150 朵花所属的具体类别。

---

### 3. Notebook 中同类数据的对比

在该代码单元格下方输出的其他几个数据集也是相同的含义：
* `Breast Cancer: (569, 30)` $\rightarrow$ **569 个样本**，每个样本有 **30 个特征**
* `Digits: (1797, 64)` $\rightarrow$ **1797 张手写数字图片**，每张图片有 **64 个像素特征**（8×8 像素展平）
* `California Housing: (20640, 8)` $\rightarrow$ **20640 个街区样本**，每个街区有 **8 个特征**

## 2. 数据集划分

In [3]:
# 基本划分
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target,
    test_size=0.3,       # 30% 作为测试集
    random_state=42,     # 固定随机种子保证可复现
    stratify=iris.target # 分层抽样，保持类别比例
)

print(f'训练集: {X_train.shape}')
print(f'测试集: {X_test.shape}')
print(f'\n训练集类别分布: {np.bincount(y_train)}')
print(f'测试集类别分布: {np.bincount(y_test)}')

训练集: (105, 4)
测试集: (45, 4)

训练集类别分布: [35 35 35]
测试集类别分布: [15 15 15]


`random_state=42` 是用来**设置随机数种子（Random Seed）**，作用是**保证代码每次运行划分出的训练集和测试集完全相同（即实验结果可复现）**。

---

### 1. 为什么需要它？

`train_test_split` 在拆分数据前，会先对数据进行**随机打乱（Shuffle）**。

* **如果不设置 `random_state`（默认是 `None`）：**
  * 每次运行代码时，系统会使用当前时间等不可控因素作为随机种子。
  * 导致**每次划分出来的数据都不一样**：比如第 1 次运行第 5 号样本分到了测试集，第 2 次运行可能就分到了训练集。
  * 这样会带来问题：模型分数忽高忽低，你无法确定是修改模型参数带来的提升，还是仅仅因为这次运气好分到的测试集比较简单。

* **如果设置了 `random_state=42`：**
  * 计算机产生的是“伪随机数”。一旦固定了种子（Seed），内部的随机序列和洗牌顺序就**完全固定**了。
  * 无论你在自己的电脑上运行多少次，或者发给同事在其他电脑上运行，**切分出的 `X_train`、`X_test` 都是一模一样的数据**。

---

### 2. 为什么通常填 42？有什么特殊含义吗？

* **在算法层面上没有特殊含义：**
  你可以填任意整数，比如 `random_state=0`、`random_state=1`、`random_state=123` 等等。只要填的值固定不变，效果都是一样的。
* **文化梗（彩蛋）：**
  选 `42` 是程序员和数据科学领域的一个致敬梗，源自科幻小说《银河系漫游指南》（*The Hitchhiker's Guide to the Galaxy*），在书中，超级计算机计算出的“关于生命、宇宙以及一切终极问题的答案”就是 **42**。因此大家习惯用 42 作为默认随机种子。

---

### 总结
* **含义**：固定伪随机数生成器的种子。
* **目的**：确保划分结果稳定、实验**可复现（Reproducible）**，方便调试和公平对比不同模型的表现。

## 3. 特征标准化

In [4]:
# StandardScaler: 均值为0，标准差为1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
X_test_scaled = scaler.transform(X_test)        # 只 transform！

print('标准化前:')
print(f'  均值: {X_train.mean(axis=0).round(2)}')
print(f'  标准差: {X_train.std(axis=0).round(2)}')

print('\n标准化后:')
print(f'  均值: {X_train_scaled.mean(axis=0).round(4)}')
print(f'  标准差: {X_train_scaled.std(axis=0).round(4)}')

标准化前:
  均值: [5.87 3.06 3.78 1.21]
  标准差: [0.86 0.46 1.78 0.77]

标准化后:
  均值: [ 0. -0. -0. -0.]
  标准差: [1. 1. 1. 1.]


特征标准化（Standardization，这里使用的是 Z-score 标准化）的核心目的，是**消除不同特征之间因量纲（单位）和取值范围差异带来的负面影响，让所有特征处于同一数量级上进行公平比较**。

公式为：
$$z = \frac{x - \mu}{\sigma}$$
（其中 $\mu$ 为均值，$\sigma$ 为标准差，处理后数据的均值为 0，标准差为 1）。

具体有以下几个主要目的和好处：

---

### 1. 消除量纲与数值尺度的影响（避免“大数吃小数”）
现实数据中不同特征的单位和数值范围通常差距巨大：
* 例如在房价预测中：
  * **房屋面积**：$50 \sim 300$ 平方米
  * **房间数量**：$1 \sim 5$ 间
* 如果不进行标准化，在计算距离（如欧氏距离）时，面积数值变动几十，对距离的影响会远远压过房间数量变动 $1 \sim 2$。但实际上“多一个房间”对房价的影响可能非常关键。
* 标准化后，每个特征都以“偏离均值多少个标准差”为度量，**消除了量纲差距，让模型公平对待每个特征**。

---

### 2. 加快基于梯度下降算法的收敛速度
像逻辑回归（Logistic Regression）、线性回归（Linear Regression）、神经网络（Neural Networks）等使用**梯度下降法**求解的模型：
* **未标准化时**：损失函数的等高线往往是极其扁平的椭圆形。梯度下降时参数更新容易来回震荡，收敛速度非常慢，甚至容易发散。
* **标准化后**：特征尺度一致，损失函数的等高线更接近圆形，梯度方向直接指向极小值点，**收敛速度大幅提升，训练更稳定**。

---

### 3. 保证基于“距离/相似度”的算法能够正确工作
许多机器学习算法的核心依赖于样本间的几何距离或内积计算：
* **KNN（K近邻）**：计算欧氏距离，若未标准化，数值范围大的特征会绝对主导近邻判定。
* **SVM（支持向量机）**：使用 RBF 核或计算间隔超平面时，严重依赖距离计算。
* **K-Means 聚类**、**PCA（主成分分析）**：PCA 计算的是方差最大的方向，如果某个特征数值很大（方差天然很大），PCA 会误认为该特征最重要，从而导致投影失真。

---

### 4. 让正则化（Regularization）更加公平有效
在线性模型中通常会加入 L1（Lasso）或 L2（Ridge）正则化惩罚项：
$$\lambda \sum |w_i| \quad \text{或} \quad \lambda \sum w_i^2$$
* 如果未标准化，数值范围很小的特征对应权重 $w_i$ 天然会很大，该权重就会被正则化项“过度惩罚”；而数值大的特征权重很小，几乎不受惩罚。
* 标准化能确保**正则化对每个特征的权重惩罚尺度一致**。

---

### 补充细节：为什么代码中 `X_test` 必须“只 transform”？

你的代码里有一句关键注释：
```python
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
X_test_scaled = scaler.transform(X_test)        # 只 transform！
```
* **`fit` 的含义**：计算当前数据的均值 $\mu$ 和标准差 $\sigma$。
* **`transform` 的含义**：用已知的 $\mu$ 和 $\sigma$ 进行公式计算：$(X - \mu) / \sigma$。
* **目的（防止数据泄露 Data Leakage）**：
  测试集模拟的是“未来未知的真实数据”。模型在训练阶段**绝不能提前获知测试集的任何统计分布信息（如测试集的均值/标准差）**。因此必须严格使用**训练集学到的均值和标准差**来转换测试集。

In [5]:
# MinMaxScaler: 缩放到 [0, 1]
minmax = MinMaxScaler()
X_train_minmax = minmax.fit_transform(X_train)

print(f'MinMax 范围: [{X_train_minmax.min():.2f}, {X_train_minmax.max():.2f}]')

MinMax 范围: [0.00, 1.00]


## 4. 缺失值处理

In [ ]:
# 创建带缺失值的数据
X_with_nan = X_train.copy().astype(float)

print(X_with_nan[:5])  # 显示前5行数据

rng = np.random.RandomState(42)
mask = rng.random(X_with_nan.shape) < 0.1  # 10% 缺失
X_with_nan[mask] = np.nan

print(X_with_nan[:5])  # 显示前5行数据
print(f'缺失值数量: {np.isnan(X_with_nan).sum()}')

# 用中位数填充
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_with_nan)

print(f'填充后缺失值: {np.isnan(X_imputed).sum()}')
print(X_imputed[:5])  # 显示前5行填充后的数据

[[5.1 2.5 3.  1.1]
 [6.2 2.2 4.5 1.5]
 [5.1 3.8 1.5 0.3]
 [6.8 3.2 5.9 2.3]
 [5.7 2.8 4.1 1.3]]
[[5.1 2.5 3.  1.1]
 [6.2 2.2 nan 1.5]
 [5.1 3.8 nan 0.3]
 [6.8 3.2 5.9 2.3]
 [5.7 2.8 4.1 1.3]]
缺失值数量: 51
填充后缺失值: 0
[[5.1  2.5  3.   1.1 ]
 [6.2  2.2  4.15 1.5 ]
 [5.1  3.8  4.15 0.3 ]
 [6.8  3.2  5.9  2.3 ]
 [5.7  2.8  4.1  1.3 ]]


## 5. 类别编码

In [9]:
# Label Encoding (有序类别)
le = LabelEncoder()
colors = ['red', 'blue', 'green', 'red', 'blue']
encoded = le.fit_transform(colors)
print(f'Label Encoding: {colors} -> {encoded}')

# One-Hot Encoding (无序类别)
ohe = OneHotEncoder(sparse_output=False)
encoded_ohe = ohe.fit_transform(np.array(colors).reshape(-1, 1))
print(f'\nOne-Hot Encoding:')
print(f'  类别: {ohe.categories_}')
print(f'  编码:\n{encoded_ohe}')

Label Encoding: ['red', 'blue', 'green', 'red', 'blue'] -> [2 0 1 2 0]

One-Hot Encoding:
  类别: [array(['blue', 'green', 'red'], dtype='<U5')]
  编码:
[[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]


这段代码的作用是演示**独热编码（One-Hot Encoding）**，用于将**无顺序关系的离散类别型特征**转换为机器学习模型能够理解的**数值型矩阵**。

在运行此代码前，上下文定义的输入数据是：
```python
colors = ['red', 'blue', 'green', 'red', 'blue']
```

---

### 一、逐行代码详解

#### 1. `ohe = OneHotEncoder(sparse_output=False)`
* **作用**：实例化一个独热编码器。
* **参数 `sparse_output=False`**：
  * 默认情况下，sklearn 为了节省内存会返回“稀疏矩阵（Sparse Matrix）”。
  * 设置为 `False` 表示**直接返回一个标准的 NumPy 二维稠密数组（Dense Array）**，方便直接查看和计算（旧版本 sklearn 中该参数名为 `sparse=False`）。

#### 2. `encoded_ohe = ohe.fit_transform(np.array(colors).reshape(-1, 1))`
这行包含两个关键步骤：
* **`np.array(colors).reshape(-1, 1)`（维度调整）**：
  * `colors` 本身是一个一维列表，长度为 5。
  * `OneHotEncoder` 规定输入的特征必须是**二维数组**，形状为 `(样本数, 特征数)`。
  * `.reshape(-1, 1)` 将一维数据变成了 $5 \times 1$ 的二维列向量：
    $$\begin{bmatrix} \text{'red'} \\ \text{'blue'} \\ \text{'green'} \\ \text{'red'} \\ \text{'blue'} \end{bmatrix}$$
* **`fit_transform(...)`（拟合并转换）**：
  * **`fit`**：扫描数据，统计出一共有多少个互不相同的类别（按字母顺序排为：`['blue', 'green', 'red']`）。
  * **`transform`**：将每个样本的颜色替换为对应的“0/1 向量”。

#### 3. `print(f'类别: {ohe.categories_}')` 与 `print(f'编码:\n{encoded_ohe}')`
* **`ohe.categories_`**：输出编码器学到的类别列表（按字典序排为 `['blue', 'green', 'red']`）。
* **`encoded_ohe`**：输出最终生成的独热编码矩阵。

---

### 二、输出结果解析

运行后的输出如下：
```text
One-Hot Encoding:
  类别: [array(['blue', 'green', 'red'], dtype='<U5')]
  编码:
[[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
```

#### 编码规则（三列分别代表）：
* **第 1 列**：是否是 `blue`
* **第 2 列**：是否是 `green`
* **第 3 列**：是否是 `red`

每个样本的对应关系如下：
| 原始颜色 | blue | green | red | 独热向量 |
| :--- | :---: | :---: | :---: | :---: |
| `'red'` | 0 | 0 | **1** | `[0., 0., 1.]` |
| `'blue'` | **1** | 0 | 0 | `[1., 0., 0.]` |
| `'green'` | 0 | **1** | 0 | `[0., 1., 0.]` |
| `'red'` | 0 | 0 | **1** | `[0., 0., 1.]` |
| `'blue'` | **1** | 0 | 0 | `[1., 0., 0.]` |

---

### 三、为什么要用 One-Hot Encoding？（与 Label Encoding 对比）

在上一个单元格中，如果使用 `LabelEncoder`：
* `red -> 2`、`blue -> 0`、`green -> 1`
* 这会带来一个严重问题：**人为引入了虚假的大小关系**（例如 $2 > 1 > 0$，模型会误认为 $\text{red} > \text{green} > \text{blue}$，或者 $\text{blue} + \text{green} \approx \text{red}$）。
* 对于线性模型、逻辑回归、SVM、神经网络等算法，这种虚假的数值大小会严重误导模型学习。

**One-Hot Encoding 的优势**：
* 为每个类别分配独立的一列，向量之间的欧氏距离全部相等（两两距离均为 $\sqrt{2}$）。
* **消除了类别之间的虚假大小或顺序关系**，非常适合颜色、国家、性别等**名义类别（无序类别）**变量。

## 6. 完整预处理流程

In [10]:
# 用 DataFrame 演示完整流程
df = pd.DataFrame({
    'age': [25, 30, np.nan, 45, 35, 28, np.nan, 50],
    'income': [50000, 60000, 70000, np.nan, 80000, 55000, 65000, 90000],
    'city': ['Beijing', 'Shanghai', 'Beijing', 'Guangzhou', 'Shanghai', 'Beijing', 'Guangzhou', 'Shanghai'],
    'purchased': [0, 1, 0, 1, 1, 0, 1, 1]
})

print('原始数据:')
print(df)

# Step 1: 分离特征和标签
X = df.drop('purchased', axis=1)
y = df['purchased']

# Step 2: 处理缺失值
num_cols = ['age', 'income']
imputer = SimpleImputer(strategy='median')
X[num_cols] = imputer.fit_transform(X[num_cols])

# Step 3: 类别编码
X = pd.get_dummies(X, columns=['city'], drop_first=True)

# Step 4: 标准化
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print('\n处理后:')
print(X_scaled.round(2))

原始数据:
    age   income       city  purchased
0  25.0  50000.0    Beijing          0
1  30.0  60000.0   Shanghai          1
2   NaN  70000.0    Beijing          0
3  45.0      NaN  Guangzhou          1
4  35.0  80000.0   Shanghai          1
5  28.0  55000.0    Beijing          0
6   NaN  65000.0  Guangzhou          1
7  50.0  90000.0   Shanghai          1

处理后:
    age  income  city_Guangzhou  city_Shanghai
0 -1.22   -1.38           -0.58          -0.77
1 -0.59   -0.56           -0.58           1.29
2 -0.28    0.26           -0.58          -0.77
3  1.28   -0.15            1.73          -0.77
4  0.03    1.07           -0.58           1.29
5 -0.84   -0.97           -0.58          -0.77
6 -0.28   -0.15            1.73          -0.77
7  1.91    1.89           -0.58           1.29


这段代码演示了一个**经典且完整的表格数据预处理全流程（Pipeline）**。在工业界和实际项目中，拿到原始数据后送入机器学习模型前，通常都需要经历这几个核心步骤。

下面分步为您详细解析：

---

### 0. 构造示例数据集
```python
df = pd.DataFrame({
    'age': [25, 30, np.nan, 45, 35, 28, np.nan, 50],
    'income': [50000, 60000, 70000, np.nan, 80000, 55000, 65000, 90000],
    'city': ['Beijing', 'Shanghai', 'Beijing', 'Guangzhou', 'Shanghai', 'Beijing', 'Guangzhou', 'Shanghai'],
    'purchased': [0, 1, 0, 1, 1, 0, 1, 1]
})
```
* 模拟了 8 个用户的画像数据，包含典型的各类字段：
  * **数值型特征且含缺失值**：`age`（年龄，有 2 个 `NaN`）、`income`（收入，有 1 个 `NaN`）。
  * **无序类别型特征**：`city`（城市：Beijing、Shanghai、Guangzhou）。
  * **预测目标（标签）**：`purchased`（是否购买：0 或 1）。

---

### Step 1: 分离特征矩阵（X）和目标标签（y）
```python
X = df.drop('purchased', axis=1)
y = df['purchased']
```
* **作用**：
  * **`X`**：去掉目标列 `purchased`，只保留用于预测的输入特征（`age`, `income`, `city`）。
  * **`y`**：提取出监督学习要预测的真实标签。
* **原因**：所有的预处理变换（填充、编码、标准化）**只针对特征 `X` 进行**，绝不能污染预测目标 `y`。

---

### Step 2: 数值特征缺失值填充（Imputation）
```python
num_cols = ['age', 'income']
imputer = SimpleImputer(strategy='median')
X[num_cols] = imputer.fit_transform(X[num_cols])
```
* **作用**：
  * 提取数值列 `['age', 'income']`。
  * 使用中位数策略（`strategy='median'`）填充 `NaN` 缺失值。
* **为什么选中位数？**
  * 对于现实中的年龄、收入等指标，均值很容易受极大值（极端富豪等）影响产生偏移，而**中位数（Median）抗极端异常值的鲁棒性更强**。
* **执行结果**：
  * `age` 现有数据的中位数是 $32.5$（填充到第 3 行和第 7 行）。
  * `income` 现有数据的中位数是 $65000$（填充到第 4 行）。

---

### Step 3: 类别特征编码（One-Hot Encoding）
```python
X = pd.get_dummies(X, columns=['city'], drop_first=True)
```
* **作用**：
  * 使用 Pandas 的 `get_dummies` 函数对 `city` 列进行独热编码。
* **参数 `drop_first=True` 是关键精髓**：
  * `city` 有 3 个类别：`Beijing`, `Guangzhou`, `Shanghai`。
  * 正常独热编码会生成 3 列，但 `drop_first=True` 会**丢弃第 1 个类别（Beijing）**，只保留 `city_Guangzhou` 和 `city_Shanghai` 两列。
  * **为什么丢弃一列？**
    * 避免**多重共线性（虚拟变量陷阱 Dummy Variable Trap）**。
    * 如果 `city_Guangzhou == 0` 且 `city_Shanghai == 0`，那它必然就是 `Beijing`。保留 3 列会导致列之间完全线性相关，在线性回归、逻辑回归等模型中会导致矩阵求逆出错或权重不可解。

---

### Step 4: 特征标准化（Standardization）
```python
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
```
* **作用**：
  * 使用 `StandardScaler` 对整个特征表进行 Z-score 标准化（$z = \frac{x - \mu}{\sigma}$）。
  * `scaler.fit_transform(X)` 会返回纯 NumPy 数组，所以外面套了一层 `pd.DataFrame(..., columns=X.columns)`，把原本的列名保留回来。
* **目的**：
  * `income` 的数值高达几万，而 `age` 只有几十，两个 dummy 变量只有 0 和 1。
  * 标准化将所有列转换成**均值为 0、标准差为 1** 的标准分布，让每个特征处于同一尺度，利于模型快速收敛并防止大数值主导预测。

---

### 总结预处理流程的演变

经过上述 4 步后，原始数据完成了华丽转身：

| 阶段 | 数据形态 | 解决的问题 |
| :--- | :--- | :--- |
| **原始数据** | 包含 NaN、字符串文本、数值量纲极大不同 | 绝大多数 ML 算法直接运行报错 |
| **Step 1** | 特征和标签分离 | 明确自变量与因变量 |
| **Step 2** | 数值列无缺失值 | 解决数据缺失导致模型无法计算问题 |
| **Step 3** | 字符串转为 0/1 哑变量 | 解决数学模型无法输入字符串问题 |
| **Step 4** | 全局均值 0，方差 1 | 消除量纲差异，加速梯度收敛，提升算法精度 |

最终得到的 `X_scaled` 即可安全送入逻辑回归、SVM、神经网络等算法进行训练。

## ⚠️ 常见错误

1. **数据泄漏**: 在测试集上 fit scaler
2. **忘记分层抽样**: 类别不平衡时不用 stratify
3. **不处理缺失值**: 很多模型不支持 NaN

---
下一个 Notebook: [03_linear_regression.ipynb](03_linear_regression.ipynb)